# 09 · Silver · Site (canal e-commerce)
Tipa, converte os horários e marca as linhas do site. **Nada é removido (D07).**

| Regra | Coluna | Origem |
|---|---|---|
| Pedido, item ou evento reenviado pela exportação incremental | `versao_atual` / `duplicada` | Contrato: CDC por `updated_at` |
| Pedido que conta como venda | `valido` = versão atual e `status = 'FINALIZADO'` | Contrato, seção Status |
| Item de venda | `valido` = item atual, pedido válido, SKU do catálogo e quantidade positiva | D12, D18 |
| Evento de navegação | `valido` = não duplicado e tipo conhecido | Contrato, seção 3 |

**Horários:** chegam em UTC (`...Z`). A coluna `data` usa o horário de Brasília, igual aos outros canais: uma compra às 22h do dia 10 é venda do dia 10, e não do dia 11.

**Equivalente no BigQuery:** `CAST(x AS TIMESTAMP)` lê o `Z`; horário local com `DATETIME(ts, 'America/Sao_Paulo')` (no Databricks, `from_utc_timestamp`); `DECIMAL` → `NUMERIC`. `ROW_NUMBER()` é igual nos dois.

### 1. Pedidos
Um cancelamento chega como **nova versão** do mesmo `pedido_id`, com `updated_at` mais recente. A versão atual é a de `updated_at` mais recente; as anteriores ficam marcadas.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.site_pedidos AS
WITH tipada AS (
  SELECT pedido_id, usuario_hash, status, forma_pagamento,
         CAST(subtotal AS DECIMAL(12,2))     AS subtotal,
         CAST(frete AS DECIMAL(12,2))        AS frete,
         CAST(valor_total AS DECIMAL(12,2))  AS valor_total,
         CAST(finalizado_em AS TIMESTAMP)    AS finalizado_em_utc,
         CAST(cancelado_em AS TIMESTAMP)     AS cancelado_em_utc,
         CAST(updated_at AS TIMESTAMP)       AS updated_at,
         origem, _arquivo_origem
  FROM horizonte.bronze.site_pedidos
),
marcada AS (
  SELECT *,
         from_utc_timestamp(finalizado_em_utc, 'America/Sao_Paulo') AS finalizado_em,
         from_utc_timestamp(cancelado_em_utc, 'America/Sao_Paulo')  AS cancelado_em,
         ROW_NUMBER() OVER (PARTITION BY pedido_id
                            ORDER BY updated_at DESC, _arquivo_origem DESC) = 1 AS versao_atual,
         status NOT IN ('FINALIZADO', 'CANCELADO')                          AS status_invalido,
         valor_total <> subtotal + frete                                    AS valor_inconsistente
  FROM tipada
)
SELECT pedido_id, usuario_hash, status, forma_pagamento, subtotal, frete, valor_total,
       finalizado_em, CAST(finalizado_em AS DATE) AS data, cancelado_em, updated_at, origem,
       versao_atual, status_invalido, valor_inconsistente,
       versao_atual AND status = 'FINALIZADO' AND NOT valor_inconsistente AS valido,
       _arquivo_origem
FROM marcada;

### 2. Itens
O item herda a data e a validade do pedido. A categoria, o grupo e o custo vêm da `dim_produto` (catálogo v2). Receita do item = quantidade × preço unitário, **sem frete** (como no marketplace, D20).

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.site_pedido_itens AS
WITH tipada AS (
  SELECT item_id, pedido_id, sku,
         CAST(quantidade AS INT)                AS quantidade,
         CAST(preco_unitario AS DECIMAL(12,2))  AS preco_unitario,
         tamanho, cor,
         CAST(updated_at AS TIMESTAMP)          AS updated_at,
         origem, _arquivo_origem
  FROM horizonte.bronze.site_pedido_itens
),
marcada AS (
  SELECT t.*,
         ROW_NUMBER() OVER (PARTITION BY t.item_id
                            ORDER BY t.updated_at DESC, t._arquivo_origem DESC) = 1 AS item_atual,
         p.sku IS NULL                                      AS sku_sem_cadastro,
         t.quantidade IS NULL OR t.quantidade <= 0          AS quantidade_invalida,
         p.categoria, p.grupo, p.custo
  FROM tipada t
  LEFT JOIN horizonte.silver.dim_produto p ON p.sku = t.sku
)
SELECT m.item_id, m.pedido_id, ped.data, ped.usuario_hash, m.sku, m.categoria, m.grupo,
       m.quantidade, m.preco_unitario, m.tamanho, m.cor,
       CAST(m.quantidade * m.preco_unitario AS DECIMAL(12,2))  AS receita,
       CAST(m.quantidade * m.custo AS DECIMAL(12,2))           AS custo_total,
       m.updated_at, m.origem,
       m.item_atual, m.sku_sem_cadastro, m.quantidade_invalida,
       m.item_atual AND COALESCE(ped.valido, FALSE)
         AND NOT m.sku_sem_cadastro AND NOT m.quantidade_invalida  AS valido,
       m._arquivo_origem
FROM marcada m
LEFT JOIN horizonte.silver.site_pedidos ped
       ON ped.pedido_id = m.pedido_id AND ped.versao_atual;

### 3. Eventos
Eventos nunca são alterados: uma repetição do mesmo `evento_id` é reenvio, marcada como `duplicada`.

In [0]:
CREATE OR REPLACE TABLE horizonte.silver.site_eventos AS
WITH tipada AS (
  SELECT evento_id, evento,
         CAST(ocorrido_em AS TIMESTAMP)      AS ocorrido_em_utc,
         sessao_id, NULLIF(usuario_hash, '') AS usuario_hash, NULLIF(sku, '') AS sku,
         CAST(quantidade AS INT)             AS quantidade,
         CAST(valor AS DECIMAL(12,2))        AS valor,
         NULLIF(pedido_id, '')               AS pedido_id,
         origem, _arquivo_origem
  FROM horizonte.bronze.site_eventos
),
marcada AS (
  SELECT *,
         from_utc_timestamp(ocorrido_em_utc, 'America/Sao_Paulo')      AS ocorrido_em,
         ROW_NUMBER() OVER (PARTITION BY evento_id ORDER BY _arquivo_origem) > 1 AS duplicada,
         evento NOT IN ('session_start', 'view_product', 'add_to_cart',
                        'begin_checkout', 'purchase')                  AS evento_invalido
  FROM tipada
)
SELECT evento_id, evento, sessao_id, usuario_hash, sku, quantidade, valor, pedido_id,
       ocorrido_em, CAST(ocorrido_em AS DATE) AS data, origem,
       duplicada, evento_invalido,
       NOT duplicada AND NOT evento_invalido AS valido,
       _arquivo_origem
FROM marcada;

### 4. Conferência
Valores do histórico gerado (D23). Quando a API mandar arquivos, os totais crescem, e a origem `APP` aparece.

In [0]:
SELECT
  (SELECT COUNT(*) FROM horizonte.silver.site_pedidos)                               AS pedidos,          -- 20.180
  (SELECT COUNT_IF(valido) FROM horizonte.silver.site_pedidos)                       AS pedidos_validos,  -- 19.145
  (SELECT COUNT_IF(versao_atual AND status = 'CANCELADO') FROM horizonte.silver.site_pedidos) AS cancelados, -- 1.035
  (SELECT COUNT_IF(NOT versao_atual OR status_invalido OR valor_inconsistente)
     FROM horizonte.silver.site_pedidos)                                             AS pedidos_com_problema, -- 0
  (SELECT COUNT_IF(valido) FROM horizonte.silver.site_pedido_itens)                  AS itens_validos,    -- 26.934
  (SELECT SUM(CASE WHEN valido THEN quantidade END) FROM horizonte.silver.site_pedido_itens) AS unidades, -- 31.011
  (SELECT SUM(CASE WHEN valido THEN receita END) FROM horizonte.silver.site_pedido_itens)    AS receita,  -- 2.390.036,02
  (SELECT COUNT_IF(sku_sem_cadastro) FROM horizonte.silver.site_pedido_itens)        AS sku_sem_cadastro; -- 0

In [0]:
WITH s AS (
  SELECT sessao_id,
         bool_or(evento = 'add_to_cart') AS teve_carrinho,
         bool_or(evento = 'purchase')    AS comprou
  FROM horizonte.silver.site_eventos
  WHERE valido
  GROUP BY sessao_id
)
SELECT
  (SELECT COUNT_IF(valido) FROM horizonte.silver.site_eventos)       AS eventos_validos,   -- 1.816.833
  COUNT(*)                                                           AS sessoes,           -- 669.148
  ROUND(100 * COUNT_IF(comprou) / COUNT(*), 2)                       AS conversao_pct,     -- 3,02
  ROUND(100 * COUNT_IF(teve_carrinho AND NOT comprou)
            / COUNT_IF(teve_carrinho), 1)                            AS abandono_pct       -- 69,8
FROM s;